# ⚡ QAOA Green Dispatch — Exploration Notebook

**VTU Qiskit Fall Fest 2026 · Track 6: Quantum Applications for Climate Change & Sustainability**

This notebook walks through the entire QAOA Green Dispatch pipeline:
1. Generate a realistic energy demand profile
2. Build the QUBO matrix for unit commitment
3. Construct and visualise the QAOA circuit
4. Run QAOA (noiseless)
5. Run QAOA (with hardware noise model)
6. Compare against classical baselines
7. Analyse CO₂ savings

## 0. Setup & Imports

In [ ]:
import sys, os
# Add project root to path
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.data_generator import generate_daily_profile
from src.problem import ProblemParams, QuboMatrix, build_default_problem
from src.circuit import build_cost_hamiltonian, build_qaoa_circuit, draw_circuit
from src.optimizer import run_optimization, compare_solvers
from src.classical_solvers import BruteForceSolver, GreedySolver
from src.noise_model import build_noise_model
from src.co2_calculator import calculate_metrics

print('✅ All imports successful!')

## 1. Generate Daily Energy Demand Profile

In [ ]:
N_SLOTS = 3  # 3 time slots × 3h = 9h window (manageable: 4×3 = 12 qubits)

profile = generate_daily_profile(n_slots=N_SLOTS)
print('Time slots:', profile['time_labels'])
print('Demands (kW):', profile['demands'])
print('Solar availability:', profile['solar_availability'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(profile['time_labels'], profile['demands'], color='steelblue')
axes[0].set_title('Energy Demand Profile')
axes[0].set_ylabel('Demand (kW)')
axes[0].set_xlabel('Time')

axes[1].plot(profile['time_labels'], profile['solar_availability'],
             marker='o', color='orange', linewidth=2)
axes[1].fill_between(range(N_SLOTS), profile['solar_availability'],
                     alpha=0.3, color='orange')
axes[1].set_title('Solar Availability Factor')
axes[1].set_ylabel('Availability (0-1)')
axes[1].set_xlabel('Time')
axes[1].set_xticks(range(N_SLOTS))
axes[1].set_xticklabels(profile['time_labels'])

plt.tight_layout()
plt.show()

## 2. Build the QUBO Matrix

In [ ]:
params = ProblemParams(
    n_slots=N_SLOTS,
    demands=profile['demands'],
    solar_avail=profile['solar_availability'],
    penalty_weight=10.0,
    co2_weight=0.5,
)
qubo = QuboMatrix(params).build()

Q = qubo.Q
print(f'QUBO size: {Q.shape[0]} × {Q.shape[1]} ({params.n_vars} binary variables)')
print(f'n_sources={params.n_sources}, n_slots={params.n_slots}')
print(f'Qubits needed: {params.n_vars}')

# Visualise QUBO matrix
plt.figure(figsize=(8, 7))
plt.imshow(Q, cmap='RdBu_r', aspect='auto')
plt.colorbar(label='QUBO coefficient')
plt.title(f'QUBO Matrix Q ({params.n_vars}×{params.n_vars})')
plt.xlabel('Variable index j')
plt.ylabel('Variable index i')

# Add source labels
ticks = [i * params.n_slots + params.n_slots // 2 for i in range(params.n_sources)]
plt.xticks(ticks, params.source_names, rotation=30)
plt.yticks(ticks, params.source_names)
plt.tight_layout()
plt.show()

## 3. Build and Visualise the QAOA Circuit

In [ ]:
P = 1  # QAOA depth

J, h, offset = qubo.to_ising()
print(f'Ising offset: {offset:.4f}')
print(f'h (local fields) shape: {h.shape}')
print(f'J (couplings) nnz: {np.count_nonzero(J)}')

cost_op = build_cost_hamiltonian(J, h)
print(f'\nCost Hamiltonian: {cost_op.num_qubits} qubits, {len(cost_op)} Pauli terms')

circuit, gammas, betas = build_qaoa_circuit(cost_op, p=P)
print(f'\nQAOA Circuit (p={P}):')
print(f'  Qubits : {circuit.num_qubits}')
print(f'  Depth  : {circuit.depth()}')
print(f'  Params : {len(circuit.parameters)} ({len(gammas)}γ + {len(betas)}β)')

# Draw with bound parameters for clarity
sample_params = {g: round(0.5, 2) for g in gammas}
sample_params.update({b: round(0.3, 2) for b in betas})
bound_circ = circuit.assign_parameters(sample_params)
print('\n--- Circuit Diagram (text) ---')
print(draw_circuit(bound_circ, output='text', fold=100))

## 4. Run QAOA — Noiseless Simulation

In [ ]:
print('Running QAOA (noiseless, p=2, 1024 shots) ...')
qaoa_result = run_optimization(params, qubo, p=2, shots=1024, use_noise=False, seed=42)

print(f'\n✅ QAOA Results (noiseless):')
print(f'   Best bitstring : {qaoa_result["best_bitstring"]}')
print(f'   Best QUBO cost : {qaoa_result["best_cost"]:.4f}')
print(f'   Circuit depth  : {qaoa_result["circuit_depth"]}')
print(f'   Runtime        : {qaoa_result["runtime_s"]:.2f}s')

# Show schedule
schedule = qaoa_result['schedule']
print('\n   Energy Dispatch Schedule:')
print(f'   {"Source":<12}', end='')
for t in range(params.n_slots):
    print(f'  t={t}', end='')
print()
for src, slots in schedule.items():
    print(f'   {src:<12}', end='')
    for v in slots:
        print(f'  {" ON" if v else " --"}', end='')
    print()

## 5. Measurement Counts Histogram

In [ ]:
counts = qaoa_result['all_counts']
sorted_counts = dict(sorted(counts.items(), key=lambda x: -x[1]))

top_n = 15
labels = list(sorted_counts.keys())[:top_n]
values = list(sorted_counts.values())[:top_n]

fig, ax = plt.subplots(figsize=(14, 5))
colors = ['gold' if bs == qaoa_result['best_bitstring'] else 'steelblue' for bs in labels]
bars = ax.bar(range(len(labels)), values, color=colors)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Count')
ax.set_title('QAOA Measurement Outcomes (top 15) — Gold = Best Solution')
plt.tight_layout()
plt.show()

print(f'Total unique bitstrings sampled: {len(counts)}')
print(f'Best bitstring "{qaoa_result["best_bitstring"]}" sampled {counts.get(qaoa_result["best_bitstring"], 0)} times')

## 6. Run QAOA — With Hardware Noise Model

In [ ]:
noise_model = build_noise_model()
print('Noise model gates:', noise_model.noise_instructions)

print('\nRunning QAOA (noisy, p=1, 1024 shots) ...')
qaoa_noisy = run_optimization(params, qubo, p=1, shots=1024, use_noise=True, seed=42)

print(f'\n✅ QAOA Results (noisy):')
print(f'   Best bitstring : {qaoa_noisy["best_bitstring"]}')
print(f'   Best QUBO cost : {qaoa_noisy["best_cost"]:.4f}')
print(f'   Runtime        : {qaoa_noisy["runtime_s"]:.2f}s')

# Compare noiseless vs noisy
print(f'\n   Noiseless cost : {qaoa_result["best_cost"]:.4f}')
print(f'   Noisy cost     : {qaoa_noisy["best_cost"]:.4f}')
delta = abs(qaoa_noisy['best_cost'] - qaoa_result['best_cost'])
print(f'   Cost delta     : {delta:.4f} (noise impact)')

## 7. Classical Baselines: Brute Force & Greedy

In [ ]:
print('Running brute-force solver ...')
bf_solver = BruteForceSolver(qubo, params)
bf_result = bf_solver.solve()
print(f'Brute Force: cost={bf_result["best_cost"]:.4f}, runtime={bf_result["runtime_s"]:.3f}s')

print('\nRunning greedy solver ...')
greedy_solver = GreedySolver(params)
greedy_result = greedy_solver.solve()
print(f'Greedy:      cost={greedy_result["best_cost"]:.4f}, runtime={greedy_result["runtime_s"]:.3f}s')

## 8. Full Comparison Table

In [ ]:
import pandas as pd

results_data = []
for name, res in [
    ('QAOA (noiseless)', qaoa_result),
    ('QAOA (noisy)', qaoa_noisy),
    ('Brute Force', bf_result),
    ('Greedy', greedy_result),
]:
    m = calculate_metrics(res['schedule'], params)
    results_data.append({
        'Solver': name,
        'QUBO Cost': round(res['best_cost'], 4),
        'Oper. Cost ($)': round(m['total_cost_usd'], 4),
        'CO₂ (kg)': round(m['total_co2_kg'], 4),
        'Demand Met (%)': round(m['demand_satisfaction_pct'], 1),
        'CO₂ Saved (kg)': round(m['co2_vs_diesel_savings_kg'], 4),
        'Runtime (s)': round(res['runtime_s'], 3),
})

df = pd.DataFrame(results_data).set_index('Solver')
print(df.to_string())
df

## 9. CO₂ Savings Visualisation

In [ ]:
# CO2 by source in QAOA optimal schedule
schedule = qaoa_result['schedule']
co2_by_source = {}
for i, (src, slots) in enumerate(schedule.items()):
    co2 = sum(v * params.capacities[i] * params.co2_factors[i] for v in slots)
    if co2 > 0:
        co2_by_source[src] = co2

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# CO2 Pie
if co2_by_source:
    axes[0].pie(
        list(co2_by_source.values()),
        labels=list(co2_by_source.keys()),
        autopct='%1.1f%%',
        startangle=90,
        colors=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4'],
    )
    axes[0].set_title('CO₂ Emissions by Source\n(QAOA Optimal Schedule)')
else:
    axes[0].text(0.5, 0.5, 'Zero CO₂ Schedule! 🌱',
                 ha='center', va='center', fontsize=14, color='green')
    axes[0].set_title('CO₂ Emissions by Source')

# Cost comparison bar chart
metrics_all = [
    calculate_metrics(qaoa_result['schedule'], params),
    calculate_metrics(bf_result['schedule'], params),
    calculate_metrics(greedy_result['schedule'], params),
]
solver_names = ['QAOA', 'Brute Force', 'Greedy']
costs = [m['total_cost_usd'] for m in metrics_all]
co2s  = [m['total_co2_kg'] for m in metrics_all]

x = np.arange(len(solver_names))
width = 0.35
bars1 = axes[1].bar(x - width/2, costs, width, label='Cost ($)', color='#4ECDC4')
ax2 = axes[1].twinx()
bars2 = ax2.bar(x + width/2, co2s, width, label='CO₂ (kg)', color='#FF6B6B', alpha=0.7)

axes[1].set_xlabel('Solver')
axes[1].set_ylabel('Operational Cost ($)', color='#4ECDC4')
ax2.set_ylabel('CO₂ Emissions (kg)', color='#FF6B6B')
axes[1].set_xticks(x)
axes[1].set_xticklabels(solver_names)
axes[1].set_title('Cost & CO₂: QAOA vs Classical Baselines')
axes[1].legend(loc='upper left')
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()

# Savings summary
m_qaoa = calculate_metrics(qaoa_result['schedule'], params)
print('\n🌱 QAOA Green Dispatch Savings vs All-Diesel Baseline:')
print(f'   Cost saved : ${m_qaoa["cost_vs_diesel_savings_usd"]:.4f}')
print(f'   CO₂ saved  : {m_qaoa["co2_vs_diesel_savings_kg"]:.4f} kg')
print(f'   Demand met : {m_qaoa["demand_satisfaction_pct"]:.1f}%')